# EDA-03: Distinct Value Blocks

A **block** is a group of `(column, value)` pairs — from ≥ 2 different columns — that appear in exactly the same set of rows $R$ (with $|R| \geq 2$).

For a value $v$ in column $V$:
- $R_v$ = set of row indices where $v$ appears
- $HC_v = \text{len}(v)^2 \times (|R_v| - 1)$ — hit count

A block combines all `(column, value)` pairs sharing $R_v$, with $HC_{\text{block}} = \sum HC_i$.

Blocks expose functional-dependency-like structure: values in different columns that always co-occur in exactly the same rows.

In [1]:
import time
from collections import defaultdict
from pathlib import Path

import pandas as pd

from llm_sql_phc_eval.datasets import load_datasets

DATA_DIR = Path("/Users/owhite/dev/whitechno-github/llm-sql-phc-databases")
datasets = load_datasets(DATA_DIR)
movies, products, beer, pdmx, fever, squad, bird = (
    datasets[k] for k in ("Movies", "Products", "Beer", "PDMX", "FEVER", "SQuAD", "BIRD")
)

print("Loaded:", {k: len(v) for k, v in datasets.items()})


Loaded: {'Movies': 15000, 'Products': 14890, 'Beer': 28479, 'PDMX': 10000, 'FEVER': 145449, 'SQuAD': 87599, 'BIRD': 15000}


## Helper functions

In [2]:
def _is_hashable_col(series, n_sample=200):
    for v in series.dropna().iloc[:n_sample]:
        if isinstance(v, (list, dict)):
            return False
    return True


def find_dv_blocks(df):
    """
    Find all blocks in df.

    Strategy: for each non-singleton (col, value), compute its row-index
    signature (sorted tuple of row indices). Group across all columns by
    signature; any signature shared by values from >= 2 different columns
    is a block.

    Returns list of block dicts sorted by hc descending.
    Each block: {row_count, n_cols, cols, hc, entries: [(col, val, hc), ...]}
    """
    df = df.reset_index(drop=True)
    sig_groups = defaultdict(list)

    for col in df.columns:
        if not _is_hashable_col(df[col]):
            continue
        for val, idx_arr in df.groupby(col, sort=False).indices.items():
            n = len(idx_arr)
            if n < 2:
                continue
            sig = tuple(sorted(idx_arr.tolist()))
            hc  = len(str(val)) ** 2 * (n - 1)
            sig_groups[sig].append((col, val, hc))

    blocks = []
    for sig, entries in sig_groups.items():
        cols_in_block = sorted({col for col, val, hc in entries})
        if len(cols_in_block) < 2:
            continue
        blocks.append({
            "row_count": len(sig),
            "rows":      sig,
            "n_cols":    len(cols_in_block),
            "cols":      cols_in_block,
            "hc":        sum(hc for col, val, hc in entries),
            "entries":   sorted(entries, key=lambda x: x[0]),
        })

    return sorted(blocks, key=lambda b: b["hc"], reverse=True)


def show_blocks(blocks, name, top_n=20):
    """Print summary stats and display top blocks as a DataFrame."""
    if not blocks:
        print(f"{name}: no blocks found")
        return

    ncols_dist = (
        pd.Series([b["n_cols"] for b in blocks])
        .value_counts().sort_index().to_dict()
    )
    rc = [b["row_count"] for b in blocks]
    hc = [b["hc"]        for b in blocks]

    print(f"{name}: {len(blocks):,} blocks")
    print(f"  n_cols per block : {ncols_dist}")
    print(f"  row_count        : {min(rc):,} – {max(rc):,}")
    print(f"  hc               : {min(hc):,} – {max(hc):,}")
    if len(blocks) > top_n:
        print(f"  (showing top {top_n} by hc)")
    print()

    rows = []
    for b in blocks[:top_n]:
        parts = []
        for col, val, h in b["entries"]:
            s = str(val)
            s = s[:27] + "..." if len(s) > 30 else s
            parts.append(f"{col}={s!r}")
        rows.append({
            "row_count": b["row_count"],
            "n_cols":    b["n_cols"],
            "hc":        b["hc"],
            "values":    " | ".join(parts),
        })
    display(pd.DataFrame(rows))


## Run on all datasets

In [3]:
# Run block-finding on all datasets and collect timing
all_blocks = {}
for ds_name, df in datasets.items():
    t0 = time.time()
    blocks = find_dv_blocks(df)
    elapsed = time.time() - t0
    all_blocks[ds_name] = blocks
    print(f"{ds_name:10s}: {len(blocks):5,} blocks  ({elapsed:.1f}s)")

# Combined summary table
rows = []
for ds_name, blocks in all_blocks.items():
    rows.append({
        "dataset":       ds_name,
        "n_blocks":      len(blocks),
        "max_n_cols":    max((b["n_cols"]    for b in blocks), default=0),
        "max_row_count": max((b["row_count"] for b in blocks), default=0),
        "max_hc":        max((b["hc"]        for b in blocks), default=0),
        "total_hc":      sum( b["hc"]        for b in blocks),
    })
display(pd.DataFrame(rows).set_index("dataset"))


Movies    : 1,250 blocks  (0.1s)
Products  : 2,061 blocks  (0.2s)
Beer      : 4,590 blocks  (0.2s)
PDMX      :   839 blocks  (0.3s)
FEVER     :     1 blocks  (0.6s)
SQuAD     :    60 blocks  (0.5s)
BIRD      : 3,471 blocks  (0.1s)


,n_blocks,max_n_cols,max_row_count,max_hc,total_hc
dataset,,,,,
Movies,1250,5,96,23207740,2548257262
Products,2061,6,203,1136385525,8975053972
Beer,4590,3,48,62104,11942630
PDMX,839,10,10000,22080044,40174795
FEVER,1,2,35639,15003598,15003598
SQuAD,60,2,5,1949716,5584876
BIRD,3471,3,41,1067989712,33723932494


## Movies

In [4]:
show_blocks(all_blocks["Movies"], "Movies")

Movies: 1,250 blocks
  n_cols per block : {2: 114, 3: 826, 4: 277, 5: 33}
  row_count        : 2 – 96
  hc               : 2,628 – 23,207,740
  (showing top 20 by hc)



,row_count,n_cols,hc,values
0,96,4,23207740,movieinfo='After surviving a car accid...' | m...
1,70,3,17085366,"movieinfo='Sam (Garrett Hedlund), the ...' | m..."
2,94,3,16403526,movieinfo='Depression-era bank robber ...' | m...
3,83,3,16133828,movieinfo='Two days before his wedding...' | m...
4,57,3,13693400,movieinfo='Dominic Toretto (Vin Diesel...' | m...
5,72,3,13298087,movieinfo='CIA agent Roger Ferris (Leo...' | m...
6,62,3,13275857,movieinfo='Virgil Cole (Ed Harris) and...' | m...
7,53,4,12982216,movieinfo='Ben Campbell (Jim Sturgess)...' | m...
8,53,3,12067068,movieinfo='Faced with an unresponsive ...' | m...
9,59,3,11826548,movieinfo='Faced with deportation to h...' | m...


## Products

In [5]:
show_blocks(all_blocks["Products"], "Products")

Products: 2,061 blocks
  n_cols per block : {2: 162, 3: 1896, 4: 1, 5: 1, 6: 1}
  row_count        : 2 – 203
  hc               : 32 – 1,136,385,525
  (showing top 20 by hc)



,row_count,n_cols,hc,values
0,34,3,1136385525,description='View larger Braun Silk-épil...' |...
1,50,2,685208258,description='Product Description Improve...' |...
2,4,3,451303650,description='Product Description Gillett...' |...
3,20,3,304351595,description='Product Description Norelco...' |...
4,125,3,202781168,"description=""Whether you're trying to fi..."" |..."
5,80,3,171226338,description='WHAT YOU GET: We will s erv...' |...
6,108,3,162262290,description='TOUSLED WAVES AND TEMPTING ...' |...
7,10,3,159158169,description='Product Description The Ora...' |...
8,6,3,119442470,description='Product Description Sonicar...' |...
9,37,3,109422900,description='Enhance Your Home’s Glamour...' |...


## Beer

In [6]:
show_blocks(all_blocks["Beer"], "Beer")

Beer: 4,590 blocks
  n_cols per block : {2: 4589, 3: 1}
  row_count        : 2 – 48
  hc               : 34 – 62,104
  (showing top 20 by hc)



,row_count,n_cols,hc,values
0,29,2,62104,beer/beerId='680' | beer/name='North Coast Old...
1,17,2,59792,beer/beerId='5923' | beer/name='Dogfish Head W...
2,21,2,54160,beer/beerId='52' | beer/name='Chimay Triple / ...
3,20,2,51851,beer/beerId='30838' | beer/name='Stone Old Gua...
4,15,2,49084,beer/beerId='31915' | beer/name='OHanlons Thom...
5,30,2,44370,beer/beerId='365' | beer/name='Sierra Nevada P...
6,18,2,37978,beer/beerId='52930' | beer/name='Anchor Our Sp...
7,31,2,37500,beer/beerId='10569' | beer/name='Dogfish Head ...
8,17,2,35488,beer/beerId='684' | beer/name='Flying Dog Snak...
9,20,2,35207,beer/beerId='53' | beer/name='Chimay Bleue &#4...


## PDMX

In [7]:
show_blocks(all_blocks["PDMX"], "PDMX")

PDMX: 839 blocks
  n_cols per block : {2: 632, 3: 132, 4: 34, 5: 10, 6: 12, 7: 8, 8: 9, 9: 1, 10: 1}
  row_count        : 2 – 10,000
  hc               : 8 – 22,080,044
  (showing top 20 by hc)



,row_count,n_cols,hc,values
0,8352,2,22080044,license='publicdomain' | licenseurl='https://c...
1,1648,2,4198203,license='cc-zero' | licenseurl='https://creati...
2,10000,6,1109889,hasannotations='True' | isdraft='False' | isof...
3,60,3,710124,bestarrangement='./data/17/37/QmZP7wiJvgcuTP.....
4,58,2,466944,bestarrangement='./data/14/43/QmWSoC52sYzN1n.....
5,56,2,450560,bestarrangement='./data/15/56/QmXzn9pMQ77ExK.....
6,9457,3,330960,nratings='0' | rating='0.0' | subsetrated='False'
7,41,2,322600,bestarrangement='./data/5/41/QmfR9dbf3T8QqV8.....
8,6025,2,301200,isbestuniquearrangement='False' | subsetdedupl...
9,37,2,285768,bestarrangement='./data/6/46/QmNus1Yu7HXgT4B.....


## FEVER

Note: list-valued `evidence` column is skipped.

In [8]:
show_blocks(all_blocks["FEVER"], "FEVER")

FEVER: 1 blocks
  n_cols per block : {2: 1}
  row_count        : 35,639 – 35,639
  hc               : 15,003,598 – 15,003,598



,row_count,n_cols,hc,values
0,35639,2,15003598,label='NOT ENOUGH INFO' | verifiable='NOT VERI...


## SQuAD

In [9]:
show_blocks(all_blocks["SQuAD"], "SQuAD")

SQuAD: 60 blocks
  n_cols per block : {2: 60}
  row_count        : 2 – 5
  hc               : 673 – 1,949,716
  (showing top 20 by hc)



,row_count,n_cols,hc,values
0,5,2,1949716,answer='exhibition game' | context='An exhibit...
1,3,2,694042,answer='Praxiteles' | context='The evolution o...
2,2,2,513185,answer='parliamentary republics' | context='Th...
3,2,2,437546,answer='The First Great Awakening' | context='...
4,2,2,416169,answer='77.5 million' | context='According to ...
5,2,2,279145,answer='the flow of control' | context='Progra...
6,3,2,216820,answer='American Idol' | context='As one of th...
7,2,2,175730,answer='The Archivist' | context='The Archivis...
8,2,2,149480,"answer='Book of the Month Club' | context=""Des..."
9,2,2,148081,answer='the Florida Supreme Court' | context='...


## BIRD

In [10]:
show_blocks(all_blocks["BIRD"], "BIRD")

BIRD: 3,471 blocks
  n_cols per block : {3: 3471}
  row_count        : 2 – 41
  hc               : 2,306 – 1,067,989,712
  (showing top 20 by hc)



,row_count,n_cols,hc,values
0,17,3,1067989712,Body='<p><strong>(Very) short sto...' | PostDa...
1,8,3,725572526,Body='<p>Before you set up your a...' | PostDa...
2,11,3,657076930,Body='<p><strong>Update</strong>:...' | PostDa...
3,8,3,560971327,"Body='<p>Okay, so here is my answ...' | PostDa..."
4,4,3,474997038,Body='<p>I agree completely with ...' | PostDa...
5,14,3,473324969,Body='<p>If the quantity of inter...' | PostDa...
6,8,3,431031062,Body='<p>It seems your question m...' | PostDa...
7,7,3,393954396,Body='<p>Half the battle with man...' | PostDa...
8,9,3,383648656,"Body='<p>First, there are differe...' | PostDa..."
9,41,3,362663120,Body='<p>The problem starts with ...' | PostDa...
